# Pricing Test: Paired vs. Independent, and the Regression-to-the-Mean Trap

A business raised prices 3% at a set of locations. Did revenue actually go up?

All data is simulated (see `generate_data.py`), so the true effect is known and we can check which methods get it right.

**How to use this notebook:** cells marked **TODO** are yours to complete. The setup and charts are done for you.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

part_a = pd.read_csv("data/locations_before_after.csv")
part_b = pd.read_csv("data/location_periods.csv")
part_a.head()

## Part A: Is the revenue increase statistically significant?

12 locations, average weekly revenue for the 8 weeks before and after the price increase.

In [ ]:
part_a["change"] = part_a["revenue_after"] - part_a["revenue_before"]
part_a["pct_change"] = part_a["change"] / part_a["revenue_before"] * 100

fig, ax = plt.subplots(figsize=(6, 5))
for _, row in part_a.iterrows():
    ax.plot([0, 1], [row["revenue_before"], row["revenue_after"]], marker="o", color="tab:blue", alpha=0.6)
ax.set_xticks([0, 1], ["Before", "After"])
ax.set_ylabel("Average weekly revenue ($)")
ax.set_title("Each line is one location")
plt.show()

part_a[["revenue_before", "revenue_after", "change", "pct_change"]].describe().round(1)

### TODO 1: Independent two-sample t-test

Treat "before" and "after" as two separate groups and run an independent t-test. Write down the p-value.

*Hint:* `stats.ttest_ind`

In [ ]:
# TODO 1: your code here


### TODO 2: Paired t-test, Wilcoxon signed-rank, and a 95% confidence interval

Now account for the fact that each location is measured twice.

1. Paired t-test on after vs. before
2. Wilcoxon signed-rank test (doesn't assume the differences are normal)
3. Mean change per location with a 95% confidence interval

*Hints:* `stats.ttest_rel`, `stats.wilcoxon`, `stats.t.interval` (use `stats.sem` of the differences)

In [ ]:
# TODO 2: your code here


### TODO 3: Explain it (2–4 sentences, your own words)

Why do the independent and paired tests give such different answers? Which one is right for this data, and why?

*Your answer:*


## Part B: The client picked its weakest locations

New scenario. 60 locations. The client gave the price increase to the **12 locations whose summer was weakest compared with their own 2-year average**. The increase started in the fall.

Columns:
- `revenue_trailing_2yr_avg`: each location's normal weekly revenue
- `revenue_summer`: the period used to pick locations
- `revenue_fall`: after the price increase
- `price_increase`: True for the 12 picked locations

In [ ]:
part_b["summer_vs_normal_pct"] = (part_b["revenue_summer"] / part_b["revenue_trailing_2yr_avg"] - 1) * 100
part_b["fall_vs_normal_pct"] = (part_b["revenue_fall"] / part_b["revenue_trailing_2yr_avg"] - 1) * 100

fig, ax = plt.subplots(figsize=(7, 5))
for treated, color, label in [(False, "tab:gray", "No price increase"), (True, "tab:orange", "Price increase")]:
    d = part_b[part_b["price_increase"] == treated]
    ax.scatter(d["summer_vs_normal_pct"], d["fall_vs_normal_pct"], color=color, label=label, alpha=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Summer vs. normal (%)")
ax.set_ylabel("Fall vs. normal (%)")
ax.set_title("How each location compared with its own normal level")
ax.legend()
plt.show()

### TODO 4: Naive before/after

For the 12 locations with the price increase, what was the % change in revenue from summer to fall? Run a paired t-test on it. If you stopped here, what would you tell the client?

In [ ]:
# TODO 4: your code here


### TODO 5: Compare against locations without the increase

Calculate the summer-to-fall % change for the 48 locations without the increase, then take the difference (a difference-in-differences estimate).

In [ ]:
# TODO 5: your code here


### TODO 6: Compare against each location's own normal level

Compare fall revenue with each location's trailing 2-year average, for both groups. What's the estimated effect now?

In [ ]:
# TODO 6: your code here


### TODO 7: Which method got it right? (your own words)

In this simulation the price increase has **no effect at all** (`TRUE_PRICE_EFFECT_B = 0.0` in `generate_data.py`).

Which method came closest? Why did the other two show a "lift"? What would you tell the client before putting any number in a deck?

*Your answer:*


### TODO 8 (optional): Give the price increase a real effect

Set `TRUE_PRICE_EFFECT_B = 0.03` in `generate_data.py`, run `python generate_data.py`, and rerun Part B. Which method recovers about 3%?

## Takeaways

<!-- TODO: 3 bullets, in your own words -->
